<a href="https://colab.research.google.com/github/dtc-oss/python_hw/blob/hw_3/hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 3. Исследовательский анализ данных Titanic

**Цель:** изучить датасет пассажиров «Титаника», исследовать выживаемость, имена пассажиров, возраст и размещение по каютам.

Ноутбук использует файл `train.csv`, расположенный рядом.

In [1]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

#RANDOM_STATE = 42

def find_data_file(filename: str) -> Path:
    """Ищет файл рядом с ноутбуком/в текущей папке и в стандартной папке задания."""
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path.cwd().parent / filename,
        Path('/mnt/data') / filename,
        Path('/mnt/data/titanic_homework') / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path.resolve()
    searched = '\\n'.join(f' - {path}' for path in candidates)
    raise FileNotFoundError(
        f'Не найден файл {filename}. Поместите его рядом с ноутбуком. Проверены пути:\\n{searched}'
    )

## 1. Загрузка датасета

In [2]:
data_path = find_data_file('train.csv')
df = pd.read_csv(data_path)
print(f'Файл: {data_path}')
print(f'Размер датасета: {df.shape[0]:,} строк × {df.shape[1]} столбцов')
display(df.head())

Файл: /content/train.csv
Размер датасета: 891 строк × 12 столбцов


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Основная информация о данных

In [3]:
print('Структура и типы данных:')
df.info()

print('\nКоличество пропусков по столбцам:')
missing = df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing)

print('Средняя и прочая статистика числовых столбцов:')
display(df.describe().T.round(2))

print('Средняя и прочая статистика всех столбцов, включая категориальные:')
display(df.describe(include='all').T)

print(f'Число полных дубликатов строк: {df.duplicated().sum()}')

Структура и типы данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Количество пропусков по столбцам:


,missing_count,missing_percent
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22
PassengerId,0,0.00
Name,0,0.00
Pclass,0,0.00
Survived,0,0.00
Sex,0,0.00
Parch,0,0.00
SibSp,0,0.00


Средняя и прочая статистика числовых столбцов:


,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.00,257.35,1.00,223.50,446.00,668.5,891.00
Survived,891.0,0.38,0.49,0.00,0.00,0.00,1.0,1.00
Pclass,891.0,2.31,0.84,1.00,2.00,3.00,3.0,3.00
Age,714.0,29.70,14.53,0.42,20.12,28.00,38.0,80.00
SibSp,891.0,0.52,1.10,0.00,0.00,0.00,1.0,8.00
Parch,891.0,0.38,0.81,0.00,0.00,0.00,0.0,6.00
Fare,891.0,32.20,49.69,0.00,7.91,14.45,31.0,512.33


Средняя и прочая статистика всех столбцов, включая категориальные:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Dooley, Mr. Patrick",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


Число полных дубликатов строк: 0


## 3. Процент выживаемости по классу пассажира

In [4]:
survival_by_class = (
    df.groupby('Pclass', observed=True)['Survived']
      .agg(passengers='size', survivors='sum', survival_rate='mean')
      .reset_index()
)
survival_by_class['survival_percent'] = (100 * survival_by_class['survival_rate']).round(2)
display(survival_by_class[['Pclass', 'passengers', 'survivors', 'survival_percent']])

print('Процент выживаемости показывает долю пассажиров данного класса со значением Survived = 1.')

,Pclass,passengers,survivors,survival_percent
0,1,216,136,62.96
1,2,184,87,47.28
2,3,491,119,24.24


Процент выживаемости показывает долю пассажиров данного класса со значением Survived = 1.


## 4. Самые популярные мужские и женские имена

В `Name` записаны фамилия, титул и имя. Имя извлекается из части после титула. Для женских имён, указанных в скобках (например, девичья фамилия/имя), берётся первый токен в скобках.

In [5]:
def extract_first_name(row):
    full_name = str(row['Name'])
    sex = row['Sex']

    # В некоторых женских записях собственное имя находится в скобках.
    if sex == 'female':
        in_parentheses = re.search(r'\(([^()]*)\)', full_name)
        if in_parentheses:
            words = re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)?", in_parentheses.group(1))
            if words:
                return words[0]

    # Обычный формат: "Surname, Title. Firstname Othernames".
    after_title = re.search(r"\.\s*([A-Za-z][A-Za-z'-]*)", full_name)
    return after_title.group(1) if after_title else np.nan


df['FirstName'] = df.apply(extract_first_name, axis=1)

def show_popular_names(data, grouping_label):
    counts = data['FirstName'].dropna().value_counts()
    if counts.empty:
        print(f'{grouping_label}: имена не найдены')
        return
    top_count = counts.max()
    tied_names = sorted(counts[counts == top_count].index.tolist())
    print(f'{grouping_label}: {", ".join(tied_names)} — {top_count} пассажир(ов) на каждое имя')

for sex, label in [('male', 'Самое популярное мужское имя'), ('female', 'Самое популярное женское имя')]:
    show_popular_names(df.loc[df['Sex'] == sex], label)

print('\nТоп 5 извлечённых имён по полу:')
name_counts = (
    df.dropna(subset=['FirstName'])
      .groupby(['Sex', 'FirstName']).size()
      .rename('count').reset_index()
      .sort_values(['Sex', 'count', 'FirstName'], ascending=[True, False, True])
)

display(name_counts.groupby('Sex', group_keys=False).head(5).style.hide(axis='index'))

Самое популярное мужское имя: William — 35 пассажир(ов) на каждое имя
Самое популярное женское имя: Anna, Mary — 15 пассажир(ов) на каждое имя

Топ 5 извлечённых имён по полу:


Sex,FirstName,count
female,Anna,15
female,Mary,15
female,Elizabeth,11
female,Margaret,10
female,Alice,6
male,William,35
male,John,25
male,George,14
male,Charles,13
male,Thomas,13


## 5. Самое популярное мужское и женское имя в каждом классе

In [6]:
class_name_rows = []
for pclass in sorted(df['Pclass'].dropna().unique()):
    for sex in ['male', 'female']:
        subset = df.loc[(df['Pclass'] == pclass) & (df['Sex'] == sex)]
        if subset['FirstName'].dropna().empty:
            class_name_rows.append({'Pclass': pclass, 'Sex': sex, 'Most_popular_name': None, 'Count': 0})
            continue
        counts = subset['FirstName'].dropna().value_counts()
        max_count = int(counts.max())
        tied_names = sorted(counts[counts == max_count].index.tolist())
        class_name_rows.append({
            'Pclass': pclass,
            'Sex': sex,
            'Most_popular_name': ', '.join(tied_names),
            'Count': max_count,
        })
class_name_summary = pd.DataFrame(class_name_rows)
display(class_name_summary)

,Pclass,Sex,Most_popular_name,Count
0,1,male,William,11
1,1,female,"Elizabeth, Margaret",5
2,2,male,William,9
3,2,female,Elizabeth,5
4,3,male,William,15
5,3,female,Anna,9


## 6. Пассажиры старше 44 лет

In [7]:
older_passengers = df.loc[df['Age'] > 44].copy()
print(f'Пассажиров старше 44 лет: {len(older_passengers)}')
display(older_passengers.head(20))

Пассажиров старше 44 лет: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C,Engelhart
62,63,0,1,"Harris, Mr. Henry Birkhardt",male,45.0,1,0,36973,83.4750,C83,S,Henry
92,93,0,1,"Chaffee, Mr. Herbert Fuller",male,46.0,1,0,W.E.P. 5734,61.1750,E31,S,Herbert
94,95,0,3,"Coxon, Mr. Daniel",male,59.0,0,0,364500,7.2500,NaN,S,Daniel
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C,George


## 7. Пассажиры мужского пола младше 44 лет

In [8]:
younger_men = df.loc[(df['Age'] < 44) & (df['Sex'] == 'male')].copy()
print(f'Мужчин младше 44 лет: {len(younger_men)}')
display(younger_men.head(20))

Мужчин младше 44 лет: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,NaN,S,Anders
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.1250,NaN,Q,Eugene
20,21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.0000,NaN,S,Joseph
21,22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,248698,13.0000,D56,S,Lawrence
23,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.5000,A6,S,William
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S,Charles


## 8. Количество кабин в зависимости от числа пассажиров

В столбце `Cabin` часть записей содержит несколько обозначений - семьи, выкупившие смежные каюты одной бронью.

In [9]:
cabin_groups = df['Cabin'].dropna().astype(str).str.split()

# Считаем пассажиров с одинаковым набором кабин
cabin_groups = cabin_groups.apply(lambda cabins: tuple(sorted(cabins)))

passengers_per_booking = cabin_groups.value_counts()

# Сколько размещений приходилось на каждое число пассажиров
occupancy_distribution = (
    passengers_per_booking.value_counts()
    .sort_index()
    .rename_axis('passengers_per_booking')
    .reset_index(name='number_of_bookings')
)

display(occupancy_distribution)

,passengers_per_booking,number_of_bookings
0,1,101
1,2,38
2,3,5
3,4,3


## Вывод

In [10]:
print('Итоги:')
print(f'- В датасете {len(df)} пассажиров и {df.shape[1]} исходных признаков.')
print(f'- Наибольшая выживаемость по классу: {int(survival_by_class.loc[survival_by_class["survival_rate"].idxmax(), "Pclass"])} класс.')
print('- Поле Cabin заполнено не у всех пассажиров, поэтому анализ кают описывает только доступные обозначения.')

Итоги:
- В датасете 891 пассажиров и 13 исходных признаков.
- Наибольшая выживаемость по классу: 1 класс.
- Поле Cabin заполнено не у всех пассажиров, поэтому анализ кают описывает только доступные обозначения.
